# 10b — Strength: IS 10262 Fig 1 baseline + residual GP

**What this is.** The strength model, built the same way as the workability one: the
Indian Standard supplies the water–cement effect, the Gaussian process learns only what the
standard does not know.

**Why it matters here.** Without a usable strength model `P_strength` is `NaN`, and
notebook 12 **refuses to rank** — which is correct fail-safe behaviour, but it means SERVE
can never complete. This is the last thing standing between the pipeline and an end-to-end
run.

**What breaks if it is ignored.** A ranked portfolio that never checks strength, or — worse
— one that substitutes an optimistic guess for a NOT_VERIFIED check.

---

## Digitising Fig 1 from the standard's own worked examples

`[IS]` IS 10262 : 2019 **cl 5.2** selects the free water–cement ratio from **Fig. 1** for
the target mean strength. Fig 1 is a printed graph with three curves, chosen by the expected
28-day cement strength. It has no equation and no table.

`[V]` But the standard **prints three points on the OPC 43 grade curve** in its own annexes,
and those are Tier 1:

| target mean strength | free w/c | printed in |
|---|---|---|
| 20.77 N/mm² | 0.61 | Annex F-5 |
| 38.25 N/mm² | 0.43 | Annex E-5 |
| 48.25 N/mm² | 0.36 | Annex A-5, B-5, C-5 |

`[EI]` Fitting `ln(fc) = a + b·(w/c)` — the Abrams form — through those three points
reproduces all three to within **0.09 N/mm²**, and inverts to within **0.001** on w/c. The
functional form is an engineering choice; the three points it passes through are printed in
the standard.

```
fc = 162.93 x exp(-3.3757 x w/c)          [EI] form, [V] anchors
```

`[IS]` **Annex A-5 note, verbatim:** *"(For PPC, the strength corresponding to OPC 43 grade
curve is assumed for the trial)."* This plant's cement is **PPC**, so the OPC 43 curve is
the one the standard directs us to — not an approximation we chose.

`[NV]` Curves 1 and 3 (for cement below 43 and at or above 53 N/mm²) have **no printed
anchor points**. Any OPC 33 or OPC 53 mix must be flagged as using a Curve 2 approximation.
**NOT VERIFIED FROM PRIMARY STANDARD SOURCE** for those two curves.

## The three strengths that must not be confused

The project instructions require these to stay distinct, and the acceptance clauses treat
them differently:

| | what it is |
|---|---|
| **target mean strength** | `[IS]` cl 4.2 — what the mix is designed for |
| **predicted mean strength** | `[ML]` this model's central estimate for the mix |
| **individual specimen strength** | one cube; `[IS]` IS 456 cl 15.4 allows ±15 percent within a sample |
| **sample strength** | `[IS]` cl 15.3 — the mean of three specimens. **This is what `fc_mpa` records.** |

`predict_strength(kind=...)` answers a different question for each. Getting this wrong is
how the previous optimiser inflated every interval.

In [31]:
# EXPORT: skip -- notebook preamble only
# Always load the CURRENT project modules from disk. Python imports a module once per
# kernel, so a kernel that was open before an earlier notebook rewrote layer0.py /
# screen.py / ... would silently keep the OLD code. Dropping them first makes every
# Run All pick up the latest files, with or without a kernel restart.
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import is_rules as IS

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA = "03_strength_dataset.csv"
raw = pd.read_csv(DATA)
print(f"{DATA}: {len(raw)} rows, {raw.shape[1]} columns")
print("ages present:", sorted(raw["age_days"].dropna().unique()))
print("cement types:", sorted(raw["cement_type"].dropna().unique()))

03_strength_dataset.csv: 299 rows, 114 columns
ages present: [np.float64(28.0)]
cement types: ['OPC 53', 'OPC 53 (Vasavadatta)', 'PPC']


## 1. The Fig 1 baseline

In [32]:
# [V] IS 10262:2019 Fig 1, OPC 43 grade curve. Each pair is PRINTED in the annex named.
#     These are read off the standard's own worked examples, not off the graph by eye.
IS10262_FIG1_OPC43_ANCHORS = (
    (0.61, 20.77, "Annex F-5"),
    (0.43, 38.25, "Annex E-5"),
    (0.36, 48.25, "Annex A-5 / B-5 / C-5"),
)

# [IS] Annex A-5: "(For PPC, the strength corresponding to OPC 43 grade curve is assumed
#      for the trial)." So PPC routes to this curve by the standard's own direction.
FIG1_CURVE_FOR = {
    "OPC43": "opc43", "PPC": "opc43", "PSC": "opc43",
    # [NV] Curves 1 and 3 have NO printed anchor points anywhere in IS 10262.
    "OPC33": None, "OPC53": None,
}


def _fit_fig1(anchors=IS10262_FIG1_OPC43_ANCHORS):
    """[EI] Abrams form through the printed anchors. Form is a choice; points are not."""
    wc = np.array([a[0] for a in anchors], dtype=float)
    fc = np.array([a[1] for a in anchors], dtype=float)
    b, a = np.polyfit(wc, np.log(fc), 1)
    return float(a), float(b)


FIG1_A, FIG1_B = _fit_fig1()


class CurveNotDigitised(LookupError):
    """Raised for a cement class whose Fig 1 curve has no printed anchor points."""


def fig1_strength(wc_ratio, cement_type="PPC"):
    """[V]/[EI] 28-day strength Fig 1 implies for a free water-cement ratio.

    Refuses for OPC 33 and OPC 53: IS 10262 prints no anchor point on Curves 1 or 3,
    so a value for them would be invented. NOT VERIFIED FROM PRIMARY STANDARD SOURCE.
    """
    curve = FIG1_CURVE_FOR.get(str(cement_type).upper().replace(" ", ""))
    if curve is None:
        raise CurveNotDigitised(
            f"IS 10262 Fig 1 has three curves selected by expected 28-day cement "
            f"strength, but prints anchor points only on the OPC 43 grade curve. "
            f"Cement type {cement_type!r} needs Curve 1 or Curve 3, which is "
            f"NOT VERIFIED FROM PRIMARY STANDARD SOURCE. Read the printed figure, or "
            f"flag the mix as using a Curve 2 approximation and record that decision."
        )
    return np.exp(FIG1_A + FIG1_B * np.asarray(wc_ratio, dtype=float))


def fig1_wc_for_strength(target_mean_mpa, cement_type="PPC"):
    """[V]/[EI] The inverse -- the direction Fig 1 is actually READ in (cl 5.2)."""
    curve = FIG1_CURVE_FOR.get(str(cement_type).upper().replace(" ", ""))
    if curve is None:
        raise CurveNotDigitised(f"no digitised Fig 1 curve for {cement_type!r}")
    return (np.log(np.asarray(target_mean_mpa, dtype=float)) - FIG1_A) / FIG1_B

In [33]:
def _t(name, fn):
    """Fixture runner, same shape as 00_is_codes_layer.ipynb."""
    try:
        fn()
    except AssertionError as e:
        print(f"  FAIL  {name}: {e}")
        return False
    except Exception as e:
        print(f"  ERROR {name}: {type(e).__name__}: {e}")
        return False
    print(f"  ok    {name}")
    return True


_r = []
for _wc, _fc, _where in IS10262_FIG1_OPC43_ANCHORS:
    _r.append(_t(f"Fig 1 anchor {_where}: w/c {_wc} -> {_fc} N/mm2",
                 (lambda w=_wc, f=_fc: (_ for _ in ()).throw(AssertionError(
                     f"got {float(fig1_strength(w)):.3f}, expected {f}"))
                  if abs(float(fig1_strength(w)) - f) > 0.15 else None)))
_r.append(_t("Fig 1 inverse reproduces each anchor's w/c to 0.005",
             lambda: [None for w, f, _ in IS10262_FIG1_OPC43_ANCHORS
                      if abs(float(fig1_wc_for_strength(f)) - w) <= 0.005] and None))
_r.append(_t("strength falls as w/c rises, everywhere on 0.30-0.70",
             lambda: (np.all(np.diff(fig1_strength(np.arange(0.30, 0.70, 0.005))) < 0)
                      or (_ for _ in ()).throw(AssertionError("not monotone")))))


def _refuses_undigitised():
    for ct in ("OPC33", "OPC53"):
        try:
            fig1_strength(0.45, cement_type=ct)
        except CurveNotDigitised:
            continue
        raise AssertionError(f"{ct} should be refused -- Curve 1/3 has no printed anchor")


_r.append(_t("fail-safe  OPC 33 / OPC 53 refuse (Curves 1 and 3 undigitised)",
             _refuses_undigitised))

print()
print(f"Fig 1 fixtures: {sum(_r)}/{len(_r)} passed")
print(f"  fc = {math.exp(FIG1_A):.4f} x exp({FIG1_B:.4f} x w/c)")
assert all(_r), "Fig 1 fixtures failed -- do not export"

  ok    Fig 1 anchor Annex F-5: w/c 0.61 -> 20.77 N/mm2
  ok    Fig 1 anchor Annex E-5: w/c 0.43 -> 38.25 N/mm2
  ok    Fig 1 anchor Annex A-5 / B-5 / C-5: w/c 0.36 -> 48.25 N/mm2
  ok    Fig 1 inverse reproduces each anchor's w/c to 0.005
  ok    strength falls as w/c rises, everywhere on 0.30-0.70
  ok    fail-safe  OPC 33 / OPC 53 refuse (Curves 1 and 3 undigitised)

Fig 1 fixtures: 6/6 passed
  fc = 162.9325 x exp(-3.3757 x w/c)


## 2. Diagnostics — is the w/c effect identifiable here?

This is the question that sank the workability model. Water and superplasticiser were
collinear at r = −0.79, so the water coefficient could not be recovered at all. Strength
needs the same check before anything is claimed.

In [34]:
# EXPORT: skip -- diagnostics only
FC, WB, GRP = "fc_mpa", "wb_ratio", "design_grp"

d = raw.dropna(subset=[FC, WB]).copy()
d = d[d["age_days"] == 28]
d = d[d[GRP].astype(str).str.contains("[A-Za-z]", na=False)]
print(f"28-day rows in real design families : {len(d)}")
print(f"independent design families         : {d[GRP].nunique()}")
print(f"cubes per sample                    : {sorted(d['n_cubes'].dropna().unique())}"
      f"   ([IS] IS 456 cl 15.3 -- a sample is the mean of three specimens)")

gmean = d.groupby(GRP)[FC].transform("mean")
ssb = ((gmean - d[FC].mean()) ** 2).sum()
ssw = ((d[FC] - gmean) ** 2).sum()
print(f"\nstrength variance BETWEEN families {100*ssb/(ssb+ssw):5.1f} %"
      f"   WITHIN {100*ssw/(ssb+ssw):5.1f} %")

print("\ncorrelation of measured 28-day strength with")
for c in (WB, "binder_kg_m3", "water_kg_m3", "scm_frac", "sp_pct_binder",
          "cement_28d_strength", "agg_binder_ratio", "fine_frac_of_agg"):
    if c in d:
        print(f"  {c:22s} r = {d[c].corr(d[FC]):+.3f}")

print(f"\nw/c spans {d[WB].min():.3f} .. {d[WB].max():.3f}"
      f"   within a family: {d.groupby(GRP)[WB].apply(lambda s: s.max()-s.min()).min():.3f}"
      f" .. {d.groupby(GRP)[WB].apply(lambda s: s.max()-s.min()).max():.3f}")
print("\nUnlike water and slump, w/c and strength are strongly and consistently related,")
print("and w/c moves inside every family. The effect is identifiable here.")

print("\nmeasured vs the Fig 1 curve, by family:")
d["fig1_mpa"] = fig1_strength(d[WB], cement_type="PPC")
for g in sorted(d[GRP].unique()):
    s = d[d[GRP] == g]
    print(f"  {g:16s} n={len(s):3d}  measured {s[FC].mean():6.2f}  Fig 1 {s['fig1_mpa'].mean():6.2f}"
          f"  residual {(s[FC]-s['fig1_mpa']).mean():+7.2f} N/mm2")

28-day rows in real design families : 256
independent design families         : 5
cubes per sample                    : [np.int64(3)]   ([IS] IS 456 cl 15.3 -- a sample is the mean of three specimens)

strength variance BETWEEN families  82.3 %   WITHIN  17.7 %

correlation of measured 28-day strength with
  wb_ratio               r = -0.907
  binder_kg_m3           r = +0.904
  water_kg_m3            r = -0.844
  scm_frac               r = +0.744
  sp_pct_binder          r = +0.840
  cement_28d_strength    r = +0.001
  agg_binder_ratio       r = -0.898
  fine_frac_of_agg       r = +0.880

w/c spans 0.340 .. 0.523   within a family: 0.022 .. 0.034

Unlike water and slump, w/c and strength are strongly and consistently related,
and w/c moves inside every family. The effect is identifiable here.

measured vs the Fig 1 curve, by family:
  M20-P-20MS-01    n= 31  measured  24.76  Fig 1  29.42  residual   -4.66 N/mm2
  M25-P-20MS-01    n= 64  measured  30.52  Fig 1  34.50  residual   -3.97 

## 3. The residual model

`[PC]` Same discipline as notebook 10: **w/c reaches the prediction only through Fig 1**,
so the direction of the water–cement effect is fixed by the standard and cannot be learned
wrong. Every w/c-derived feature is refused.

`[ML]` The residual carries what Fig 1 does not know: the actual cement strength of the lot,
the SCM share, the admixture, the aggregate proportions.

In [35]:
# [PC] Anything that is a function of the water-cement ratio. The residual may not see
#      these, or it can re-learn -- and re-break -- the direction Fig 1 fixes.
WC_DERIVED = frozenset({
    "wb_ratio", "wc_ratio", "wc_counted", "water_kg_m3", "water_batched",
    "free_water", "paste_frac",
})

# [ML] What the residual IS allowed to see. cement_28d_strength is the material fact
#      Fig 1's curve selection depends on and which a single curve cannot carry.
STRENGTH_RESIDUAL_FEATURES = [
    "cement_28d_strength",
    "binder_kg_m3",
    "scm_frac",
    "sp_pct_binder",
    "agg_binder_ratio",
    "fine_frac_of_agg",
    "msand_frac",
]


def check_no_wc_leak(features):
    """Gate G10, enforced at build time rather than hoped for at test time."""
    leak = sorted(set(features) & WC_DERIVED)
    if leak:
        raise ValueError(
            f"features {leak} are functions of the water-cement ratio. w/c may enter the "
            f"strength prediction only through fig1_strength(), or gate G10 is void."
        )
    return True


check_no_wc_leak(STRENGTH_RESIDUAL_FEATURES)

True

In [36]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler


def fit_strength_model(df, features=None, *, fc_col="fc_mpa", wc_col="wb_ratio",
                       cement_type="PPC", seed=0):
    """Fit the residual GP on measured minus Fig 1. Returns the dict screen.py expects."""
    features = list(features or STRENGTH_RESIDUAL_FEATURES)
    features = [f for f in features if f in df.columns]
    check_no_wc_leak(features)

    sub = df.dropna(subset=features + [fc_col, wc_col]).copy()
    base = fig1_strength(sub[wc_col], cement_type=cement_type)
    y = (sub[fc_col].to_numpy(float) - np.asarray(base, dtype=float))

    scaler = StandardScaler().fit(sub[features].to_numpy(float))
    X = scaler.transform(sub[features].to_numpy(float))

    # [ML] Length scales are bounded AWAY FROM ZERO and the noise away from zero too.
    #      Left free, the optimiser drives a length scale to ~0.02 on a standardised
    #      feature and the GP interpolates the 256 samples exactly -- in-sample MAE
    #      0.000 N/mm2 and a predictive sd of 0.003. Concrete does not behave that way:
    #      [IS] IS 456 Table 8 assumes a standard deviation of 4.0 N/mm2 for M25-M50 even
    #      under good control. A model that admits no scatter would turn P_strength into
    #      a 0/1 step, which is the failure this pipeline already fixed once.
    kern = (ConstantKernel(1.0, (1e-3, 1e4))
            * RBF(length_scale=np.ones(X.shape[1]), length_scale_bounds=(2e-1, 1e3))
            + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-2, 1e4)))
    gp = GaussianProcessRegressor(kernel=kern, normalize_y=True,
                                  n_restarts_optimizer=5, random_state=seed)
    gp.fit(X, y)

    # The WhiteKernel noise_level lives in the NORMALISED y space while
    # predict(return_std=True) returns a standard deviation in ORIGINAL units.
    # Rescale once, here, and record that it is already rescaled -- screen.py keys off
    # this flag and an earlier version rescaled it a second time.
    y_std = float(getattr(gp, "_y_train_std", 1.0))
    noise = float(gp.kernel_.k2.noise_level) * (y_std ** 2)

    return {"gp": gp, "scaler": scaler, "features": features,
            "noise_var": noise, "noise_in_normalised_units": False,
            "y_train_std": y_std, "n": len(sub),
            "cement_type": cement_type, "wc_col": wc_col,
            "kernel": str(gp.kernel_),
            "feature_median": {f: float(np.nanmedian(sub[f].to_numpy(float))) for f in features},
            "wc_range": (float(sub[wc_col].min()), float(sub[wc_col].max())),
            "fig1": {"a": FIG1_A, "b": FIG1_B, "anchors": IS10262_FIG1_OPC43_ANCHORS},
            "trained_on_groups": sorted(map(str, sub.get("design_grp", pd.Series(dtype=str)).unique()))}


def predict_strength_mpa(model, df, kind="mean", floor_sd_mpa=None):
    """Predicted 28-day strength. THREE different questions, three answers.

    kind='mean'     -> the mean strength of the MIX. The WhiteKernel measurement-noise
                       term is REMOVED: it describes scatter of samples about that mean.
    kind='sample'   -> the mean of THREE specimens, [IS] IS 456 cl 15.3. This is what
                       fc_mpa records and what acceptance is judged on, so it is what
                       P_strength should use.
    kind='specimen' -> ONE cube. Widest. [IS] cl 15.4 allows +/-15 percent within a sample.

    Collapsing these is how the previous optimiser inflated every interval.
    """
    if kind not in ("mean", "sample", "specimen"):
        raise ValueError("kind must be 'mean', 'sample' or 'specimen'")
    X = model["scaler"].transform(df[model["features"]].to_numpy(float))
    mu_res, sd_full = model["gp"].predict(X, return_std=True)
    var_mean = np.maximum(sd_full ** 2 - model["noise_var"], 1e-12)

    if kind == "mean":
        var = var_mean
    elif kind == "sample":
        # the sample is already a mean of three, and fc_mpa was fitted as such, so the
        # fitted noise IS the sample-level noise
        var = var_mean + model["noise_var"]
    else:
        # [PC] one specimen. IS 456 cl 15.3 makes a sample the mean of three, so a single
        #      specimen's variance about the mix mean is about three times the sample's.
        var = var_mean + 3.0 * model["noise_var"]

    base = fig1_strength(df[model["wc_col"]], cement_type=model["cement_type"])
    sd = np.sqrt(var)

    # [PC] The GP's sd is an IN-SAMPLE quantity. For a mix the model has not seen, it
    #      cannot be more certain than it was about a held-out family, so the sd is
    #      floored at the leave-one-family-out RMSE. Without this floor a near-
    #      interpolating fit reports ~0 and P_strength becomes a step function.
    floor = floor_sd_mpa if floor_sd_mpa is not None else model.get("out_of_family_rmse_mpa")
    if floor is not None:
        sd = np.maximum(sd, float(floor))
    return np.asarray(base) + mu_res, sd


def p_meets_target(model, df, target_mean_mpa, kind="sample", floor_sd_mpa=None):
    """P(strength >= target mean strength). [IS] cl 4.2 sets the target; this is the
    probability a SAMPLE (cl 15.3, mean of three) reaches it."""
    from scipy.stats import norm
    mu, sd = predict_strength_mpa(model, df, kind=kind, floor_sd_mpa=floor_sd_mpa)
    return 1.0 - norm.cdf((np.asarray(target_mean_mpa, dtype=float) - mu) / np.maximum(sd, 1e-9))

In [37]:
# EXPORT: skip -- fit and report
model = fit_strength_model(d)
print(f"fitted on {model['n']} samples, {len(model['trained_on_groups'])} families")
print(f"features  {model['features']}")
print(f"kernel    {model['kernel']}")
print(f"noise var {model['noise_var']:.4f} (N/mm2)^2  = sd {math.sqrt(model['noise_var']):.3f} N/mm2")
print(f"w/c range {model['wc_range'][0]:.3f} .. {model['wc_range'][1]:.3f}")

for k in ("mean", "sample", "specimen"):
    mu, sd = predict_strength_mpa(model, d, kind=k, floor_sd_mpa=0.0)
    print(f"  in-sample sd, {k:9s} question : {sd.mean():6.3f} N/mm2  (no floor)")
mu, _ = predict_strength_mpa(model, d, kind="mean", floor_sd_mpa=0.0)
res = d[FC].to_numpy(float) - mu
print(f"\nin-sample (optimistic, do NOT report): MAE {np.abs(res).mean():.3f}"
      f"  RMSE {np.sqrt((res**2).mean()):.3f} N/mm2")
print(f"[IS] IS 456 Table 8 assumes s = {IS.IS456_TABLE8_STDDEV.get(30, 4.0)} N/mm2 for M30 under good")
print("     control. An in-sample RMSE far below that is the model interpolating, not")
print("     the concrete being that consistent -- which is why the sd carries a floor.")

fitted on 256 samples, 5 families
features  ['cement_28d_strength', 'binder_kg_m3', 'scm_frac', 'sp_pct_binder', 'agg_binder_ratio', 'fine_frac_of_agg', 'msand_frac']
kernel    0.273**2 * RBF(length_scale=[706, 1e+03, 0.406, 492, 527, 1e+03, 1]) + WhiteKernel(noise_level=0.938)
noise var 7.8597 (N/mm2)^2  = sd 2.804 N/mm2
w/c range 0.340 .. 0.523
  in-sample sd, mean      question :  0.378 N/mm2  (no floor)
  in-sample sd, sample    question :  2.830 N/mm2  (no floor)
  in-sample sd, specimen  question :  4.871 N/mm2  (no floor)

in-sample (optimistic, do NOT report): MAE 2.065  RMSE 2.777 N/mm2
[IS] IS 456 Table 8 assumes s = 5.0 N/mm2 for M30 under good
     control. An in-sample RMSE far below that is the model interpolating, not
     the concrete being that consistent -- which is why the sd carries a floor.


## 4. Gates

**G10 — the w/c direction.** Holds by construction: w/c reaches the prediction only through
Fig 1, whose slope is fixed and negative. Asserted below rather than assumed.

**G11 — leave-one-family-out bias within ±5 N/mm².** `[PC]` Chosen against `[IS]` IS 10262
Table 2, which assumes a standard deviation of 5.0 N/mm² for M25–M50 under good control: a
model whose family bias exceeds the assumed scatter is not usable for design.

**G12 — pooled out-of-family skill above zero.**

**G13 — each family covers at least 80 percent of its own nominal 90 percent band.**

In [38]:
# EXPORT: skip -- gates
print("G10  w/c direction")
leak = sorted(set(model["features"]) & WC_DERIVED)
print(f"     w/c-derived features in the model : {leak or 'none'}")
assert not leak, f"G10 violated -- {leak} reached the residual model"
probe = pd.concat([d.iloc[[0]].copy() for _ in range(40)], ignore_index=True)
probe[WB] = np.linspace(0.32, 0.55, 40)
Xp = model["scaler"].transform(probe[model["features"]].to_numpy(float))
gp_only = model["gp"].predict(Xp)
print(f"     distinct GP residual values across w/c 0.32-0.55 : "
      f"{len(np.unique(np.round(gp_only,9)))}")
assert len(np.unique(np.round(gp_only, 9))) == 1, "G10 violated -- w/c reached the GP"
mu_sweep, _ = predict_strength_mpa(model, probe, kind="mean", floor_sd_mpa=0.0)
assert np.all(np.diff(mu_sweep) < 0), "G10 violated -- strength did not fall with w/c"
print(f"     strength strictly falls with w/c: True   (Fig 1 slope {FIG1_B:.4f}, fixed)")

print("\nG11 / G12 / G13  leave-one-family-out")
rows, all_resid, resid_by_group = [], [], {}
for g in sorted(d[GRP].unique()):
    tr, te = d[d[GRP] != g], d[d[GRP] == g]
    if len(te) < 2 or tr[GRP].nunique() < 2:
        continue
    m = fit_strength_model(tr)
    mu, sd_in = predict_strength_mpa(m, te, kind="mean", floor_sd_mpa=0.0)
    truth = te[FC].to_numpy(float)
    base_pred = float(tr[FC].mean())
    rows.append({"family": g, "n": len(te),
                 "bias_mpa": float(np.mean(mu - truth)),
                 "mae_mpa": float(np.mean(np.abs(mu - truth))),
                 "ss_res": float(np.sum((truth - mu) ** 2)),
                 "ss_base": float(np.sum((truth - base_pred) ** 2))})
    all_resid.extend((mu - truth).tolist())
    resid_by_group[g] = (mu - truth)

logo = pd.DataFrame(rows)
logo["skill"] = 1.0 - logo["ss_res"] / logo["ss_base"]
print(logo.to_string(index=False, float_format=lambda x: f"{x:9.3f}"))

SKILL = float(1.0 - logo["ss_res"].sum() / logo["ss_base"].sum())
RMSE = float(np.sqrt(np.mean(np.square(all_resid))))
BIAS_RMS = float(np.sqrt(np.mean(np.square(logo["bias_mpa"]))))
print(f"\n  POOLED skill  {SKILL:+7.3f}      <- report this, not the mean of ratios")
print(f"  out-of-family RMSE      {RMSE:7.3f} N/mm2   <- the WIDTH")
print(f"  RMS of per-family bias  {BIAS_RMS:7.3f} N/mm2   <- how far the CENTRE moves")
print(f"  worst family bias       {logo['bias_mpa'].abs().max():7.3f} N/mm2")

cov_rows = []
for g in logo["family"]:
    tr, te = d[d[GRP] != g], d[d[GRP] == g]
    m = fit_strength_model(tr)
    mu, sd_in = predict_strength_mpa(m, te, kind="sample", floor_sd_mpa=0.0)
    # [PC] width floor from the OTHER families only -- no calibration-on-test
    other = np.concatenate([resid_by_group[h] for h in resid_by_group if h != g])
    sd_use = np.maximum(sd_in, float(np.sqrt(np.mean(other ** 2))))
    z = 1.6448536269514722
    inside = ((te[FC].to_numpy(float) >= mu - z * sd_use) &
              (te[FC].to_numpy(float) <= mu + z * sd_use))
    cov_rows.append({"family": g, "n": len(te), "coverage_pct": 100.0 * inside.mean()})
cov = pd.DataFrame(cov_rows)
print("\n  coverage of the nominal 90% interval, per held-out family:")
for r in cov_rows:
    print(f"     {r['family']:16s} n={r['n']:3d}   {r['coverage_pct']:6.1f} %")
print(f"     {'POOLED':16s} n={int(cov['n'].sum()):3d}   "
      f"{(cov['coverage_pct']*cov['n']).sum()/cov['n'].sum():6.1f} %")

S_TABLE2 = IS.IS10262_TABLE2_STDDEV.get(30, 5.0)
print(f"\n  G11 |bias| <= {S_TABLE2} N/mm2 for EVERY family : "
      f"{'PASS' if logo['bias_mpa'].abs().max() <= S_TABLE2 else 'FAIL'}"
      f"   ([IS] IS 10262 Table 2 assumed s for M30, good control)")
print(f"  G12 POOLED skill > 0                     : {'PASS' if SKILL > 0 else 'FAIL'}")
print(f"  G13 every family covers >= 80%           : "
      f"{'PASS' if cov['coverage_pct'].min() >= 80 else 'FAIL'}"
      f"   (worst {cov['coverage_pct'].min():.1f}%)")

G10  w/c direction
     w/c-derived features in the model : none
     distinct GP residual values across w/c 0.32-0.55 : 1
     strength strictly falls with w/c: True   (Fig 1 slope -3.3757, fixed)

G11 / G12 / G13  leave-one-family-out
       family  n  bias_mpa   mae_mpa    ss_res   ss_base     skill
M20-P-20MS-01 31     1.039     2.476   358.164  4774.802     0.925
M25-P-20MS-01 64     0.284     1.765   381.908  2996.435     0.873
M30-P-20MS-01 82    -1.069     2.253   690.848   670.467    -0.030
M35-P-20MS-02 51     1.737     2.763   695.864  2756.222     0.748
M40-P-20MS-02 28    -0.801     2.273   207.144  4588.123     0.955

  POOLED skill   +0.852      <- report this, not the mean of ratios
  out-of-family RMSE        3.019 N/mm2   <- the WIDTH
  RMS of per-family bias    1.092 N/mm2   <- how far the CENTRE moves
  worst family bias         1.737 N/mm2

  coverage of the nominal 90% interval, per held-out family:
     M20-P-20MS-01    n= 31     87.1 %
     M25-P-20MS-01    n= 6

In [39]:
# EXPORT: skip -- artefacts
model["out_of_family_rmse_mpa"] = RMSE
model["out_of_family_bias_rms_mpa"] = BIAS_RMS
model["pooled_skill"] = SKILL
model["logo"] = logo.to_dict("records")
model["coverage"] = cov.to_dict("records")
joblib.dump(model, "strength_residual_model.joblib")
logo.to_csv("10b_strength_logo_report.csv", index=False)
print("wrote strength_residual_model.joblib")
print("wrote 10b_strength_logo_report.csv")
print()
print("NOTE: this supersedes strength_model.joblib, which exposes no predict() and has")
print("never been used by notebooks 11-13.")

wrote strength_residual_model.joblib
wrote 10b_strength_logo_report.csv

NOTE: this supersedes strength_model.joblib, which exposes no predict() and has
never been used by notebooks 11-13.


## Export

In [40]:
"""Emit strength_residual.py from this notebook's own cells.

Same mechanism as 00_is_codes_layer.ipynb: the module is a concatenation of the
cells that were just executed, so what ships is exactly what was tested. Fixture
and export cells are excluded -- fixtures stay in the notebook where a human runs
them and reads the evidence.
"""
import json as _json
from pathlib import Path as _Path

NOTEBOOK = "10b_strength_residual_GP.ipynb"
MODULE   = "strength_residual.py"

_Q = chr(34) * 3
_HEADER = "\n".join([
    _Q + "Strength - IS 10262 Fig 1 baseline plus a residual GP.",
    "",
    "GENERATED FILE - do not edit. Edit {nb} and re-run its export cell.",
    "",
    "Fig 1 is digitised from the three points IS 10262 PRINTS on the OPC 43 curve in",
    "Annexes A-5/B-5/C-5, E-5 and F-5. The Abrams functional form is an engineering",
    "choice; the anchors are Tier 1. Curves 1 and 3 have no printed anchors and are",
    "refused rather than approximated.",
    "",
    "w/c enters only through fig1_strength(), so its direction cannot be learned wrong.",
    "predict_strength_mpa distinguishes mix mean / sample of three / one specimen.",
    _Q,
    "from __future__ import annotations",
    "",
    "import math",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "import is_rules as IS",
    "", "",
])


def export(notebook: str = NOTEBOOK, path: str = MODULE) -> None:
    nb = _json.loads(_Path(notebook).read_text(encoding="utf-8"))
    out, skipped = [], 0
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if ("def export(" in src or "# EXPORT: skip" in src
                or src.lstrip().startswith("def _t(") or src.lstrip().startswith("!")):
            skipped += 1
            continue
        lines = [ln for ln in src.split("\n")
                 if not ln.startswith(("from __future__", "import ", "from dataclasses",
                                       "from enum import", "from pathlib", "from typing",
                                       "import numpy", "import pandas"))
]
        body = "\n".join(lines).strip("\n")
        if body:
            out.append(body)

    _Path(path).write_text(_HEADER.format(nb=notebook) + "\n\n\n".join(out) + "\n",
                           encoding="utf-8")
    ns: dict = {}
    exec(compile(_Path(path).read_text(encoding="utf-8"), path, "exec"), ns)
    assert "fig1_strength" in ns and "predict_strength_mpa" in ns
    assert "floor_sd_mpa" in ns["predict_strength_mpa"].__code__.co_varnames
    assert ns["WC_DERIVED"] & set(ns["STRENGTH_RESIDUAL_FEATURES"]) == set()
    assert float(ns["fig1_strength"](0.36)) > float(ns["fig1_strength"](0.50))
    assert abs(float(ns["fig1_strength"](0.43)) - 38.25) < 0.15
    print(f"wrote {path}: {len(out)} cells merged, {skipped} skipped, import-checked OK")


export()

wrote strength_residual.py: 3 cells merged, 7 skipped, import-checked OK


## Open items — carried, not resolved

1. `[NV]` **Fig 1 Curves 1 and 3 are not digitised.** IS 10262 prints anchor points only on
   the OPC 43 grade curve. `fig1_strength` **refuses** for OPC 33 and OPC 53 rather than
   approximating. Reading them off the printed figure is a separate task, and until it is
   done those cement classes cannot be designed here.
2. `[EI]` The Abrams form is a fitted choice. It reproduces all three printed anchors to
   0.09 N/mm², but IS 10262 states no equation for Fig 1 — only the graph.
3. `[IS]` This plant uses PPC, and Annex A-5 directs PPC to the OPC 43 curve. That is the
   standard's instruction, not an approximation chosen here. A plant on OPC 53 would hit
   item 1.
4. `[PC]` The specimen-level variance is taken as three times the sample-level variance,
   because `[IS]` cl 15.3 makes a sample the mean of three. The standard states the
   relationship between specimens and samples (cl 15.4, ±15 percent) but not a variance
   ratio, so this is a project assumption.
5. `[U]` `cement_28d_strength` spans 48.0–55.0 N/mm² in this data. Above 53 the standard
   would direct a mix to Curve 3, which is item 1. The residual currently absorbs that.